# A1 组（G9）验证

本 Notebook 记录对 A1 组文件的**错误校验**与**实验复现**。

- 复现环境：Windows 11，Python 3.12.10，NumPy 2.5.1，SciPy 1.18.1
- 原文声明环境：macOS，Python 3.14.6，NumPy 2.4.6，SciPy 1.18.0（跨环境复现仍一致）
- 方法：从 `A1-G9-实验.md` 抽取代码、重建 `lab_common.py`，逐实验复现标题数字。

In [1]:

import numpy as np
np.set_printoptions(precision=6, suppress=True)

# ---- lab_common 重建（A1-G9-实验.md 0.4）----
def bisect(f, a, b, tol=1e-12, maxit=200):
    fa, fb = f(a), f(b)
    if fa*fb >= 0: raise ValueError("区间端点必须异号")
    hist=[]
    for k in range(maxit):
        if (b-a)/2 <= tol: return hist, True
        m=0.5*(a+b); fm=f(m); hist.append((k,a,b,m,fm))
        if fm==0: return hist, True
        if fa*fm<0: b,fb=m,fm
        else: a,fa=m,fm
    return hist, False
def fixed_point(g, x0, tol=1e-12, maxit=200):
    x,hist=x0,[x0]
    for _ in range(maxit):
        x=g(x); hist.append(x)
        if abs(hist[-1]-hist[-2])<=tol: return np.array(hist),True
    return np.array(hist),False
def newton1d(f,df,x0,tol=1e-12,maxit=50):
    x,hist=x0,[x0]
    for _ in range(maxit):
        if df(x)==0: return np.array(hist),False
        x=x-f(x)/df(x); hist.append(x)
        if abs(f(x))<=tol: return np.array(hist),True
    return np.array(hist),False

# ---- 5.1-A 二分 ----
f=lambda x:x*x-2.0
hist,ok=bisect(f,1.0,2.0,tol=1e-12)
print("5.1-A 二分迭代次数:", len(hist), "理论下界:", int(np.ceil(np.log2(1/(2e-12)))))
g=lambda x:x**3-2*x+2
hist2,ok2=bisect(g,-2.0,0.0,tol=1e-10)
print("5.1-A x^3-2x+2 迭代次数:", len(hist2), "根≈", hist2[-1][3])

# ---- 5.1-B 不动点 ----
r2=np.sqrt(2.0)
cases={"g1":(lambda x:0.5*(x+2/x),lambda x:0.5*(1-2/x**2)),
       "g2":(lambda x:2/x,lambda x:-2/x**2),
       "g3":(lambda x:x-(x*x-2)/4,lambda x:1-x/2)}
for name,(gg,dg) in cases.items():
    xs,ok=fixed_point(gg,1.5,tol=1e-14,maxit=30)
    print(f"5.1-B {name}: ok={ok} |g'(x*)|={abs(dg(r2)):.6f}")

# ---- 5.1-C Newton ----
xs,ok=newton1d(lambda x:x*x-2,lambda x:2*x,1.0,tol=1e-15,maxit=10)
err=np.abs(xs-r2)
print("5.1-C Newton 比值:", [f"{err[k+1]/err[k]**2:.6f}" for k in range(4)], "渐近 1/(2sqrt2)=",1/(2*r2))
xs2=[2.0]
for _ in range(6):
    x=xs2[-1]; xs2.append(x-(x-1)**2/(2*(x-1)))
print("5.1-C 多重根比值:", [f"{abs(xs2[k+1]-1)/abs(xs2[k]-1):.3f}" for k in range(5)])
xs3,_=newton1d(lambda x:x**3-2*x+2,lambda x:3*x*x-2,0.0,tol=1e-15,maxit=8)
print("5.1-C 二周期:", xs3)

# ---- 5.2-B 固定步长 ----
th=np.deg2rad(30); R=np.array([[np.cos(th),-np.sin(th)],[np.sin(th),np.cos(th)]])
H=R@np.diag([1.,20.])@R.T; gphi=lambda v:H@v
def gd_fixed(alpha,itmax=20000,tol=1e-8):
    v=np.array([1.,1.])
    for k in range(itmax):
        v=v-alpha*gphi(v)
        if np.linalg.norm(gphi(v))<=tol: return k+1,True
    return itmax,False
print("5.2-B 固定步长:", {a: gd_fixed(a)[0] for a in [0.004,0.009,0.045,0.09,0.11]})

# ---- 5.3-C BFGS ----
def rosen(v):
    x,y=v; return (1-x)**2+100*(y-x*x)**2
def grosen(v):
    x,y=v; return np.array([-2*(1-x)-400*x*(y-x*x),200*(y-x*x)])
def bfgs(fn,grad,x0,itmax=2000,tol=1e-8,c1=1e-4,rho=0.5):
    x=np.asarray(x0,float).copy(); n=len(x); Hinv=np.eye(n); g=grad(x)
    for k in range(itmax):
        if np.linalg.norm(g)<=tol: return k
        d=-(Hinv@g); a,fk=1.0,fn(x)
        while fn(x+a*d)>fk+c1*a*(g@d): a*=rho
        s=a*d; x=x+s; g_new=grad(x); y=g_new-g; sy=s@y
        if sy>1e-12:
            r=1.0/sy; V=np.eye(n)-r*np.outer(s,y); Hinv=V@Hinv@V.T+r*np.outer(s,s)
        g=g_new
    return itmax
def gd(fn,grad,x0,itmax=30000,tol=1e-8):
    x=np.asarray(x0,float).copy(); g=grad(x)
    for k in range(itmax):
        if np.linalg.norm(g)<=tol: return k
        d,a,fk=-g,1.0,fn(x)
        while fn(x+a*d)>fk+1e-4*a*(g@d): a*=0.5
        x=x+a*d; g=grad(x)
    return itmax
print("5.3-C BFGS 步数:", bfgs(rosen,grosen,[-1.2,1.0]), " GD 步数:", gd(rosen,grosen,[-1.2,1.0]))

# ---- 5.4-C LM ----
rng=np.random.default_rng(42); T=np.array([0.,1.,2.,3.,4.,5.])
Y=2.0*np.exp(-0.5*T)+rng.normal(0.,0.03,T.size)
def resid(p): return p[0]*np.exp(p[1]*T)-Y
def jac(p):
    e=np.exp(p[1]*T); return np.column_stack([e,p[0]*T*e])
def Phi(p): return 0.5*resid(p)@resid(p)
def lm(p0,lam=1e-3,itmax=200,tol=1e-10):
    p=np.asarray(p0,float).copy(); nu,hist=2.0,[]
    for k in range(itmax):
        r,J=resid(p),jac(p); gg,A=J.T@r,J.T@J; ph=0.5*r@r
        if np.linalg.norm(gg)<=tol: return p,len(hist),'grad',lam
        for _ in range(60):
            d=np.linalg.solve(A+lam*np.eye(2),-gg); pred=-(gg@d+0.5*d@A@d)
            ph_new=Phi(p+d); rho=(ph-ph_new)/pred if pred>0 else -1.0
            if rho>0:
                p=p+d; lam*=max(1/3,1-(2*rho-1)**3); nu=2.0; hist.append(rho); break
            lam*=nu; nu*=2.0
        else: return p,len(hist),'no_step',lam
    return p,len(hist),'maxit',lam
for p0 in ([1.0,-0.1],[1.0,1.5]):
    p,k,why,lam=lm(p0); print(f"5.4-C LM p0={p0}: k={k} why={why} p={p}")
print("全部关键数字复核完成")


5.1-A 二分迭代次数: 39 理论下界: 39
5.1-A x^3-2x+2 迭代次数: 34 根≈ -1.7692923542344943
5.1-B g1: ok=True |g'(x*)|=0.000000
5.1-B g2: ok=False |g'(x*)|=1.000000
5.1-B g3: ok=True |g'(x*)|=0.292893
5.1-C Newton 比值: ['0.500000', '0.333333', '0.352941', '0.353522'] 渐近 1/(2sqrt2)= 0.35355339059327373
5.1-C 多重根比值: ['0.500', '0.500', '0.500', '0.500', '0.500']
5.1-C 二周期: [0. 1. 0. 1. 0. 1. 0. 1. 0.]
5.2-B 固定步长: {0.004: 4674, 0.009: 2073, 0.045: 407, 0.09: 199, 0.11: 20000}


C:\Users\29006\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\linalg\_linalg.py:2767: RuntimeWarning: overflow encountered in dot
  sqnorm = x.dot(x)
C:\Users\29006\AppData\Local\Temp\ipykernel_28144\2840240173.py:60: RuntimeWarning: overflow encountered in matmul
  H=R@np.diag([1.,20.])@R.T; gphi=lambda v:H@v
C:\Users\29006\AppData\Local\Temp\ipykernel_28144\2840240173.py:64: RuntimeWarning: invalid value encountered in subtract
  v=v-alpha*gphi(v)


5.3-C BFGS 步数: 35  GD 步数: 19435
5.4-C LM p0=[1.0, -0.1]: k=7 why=grad p=[ 2.007756 -0.510444]
5.4-C LM p0=[1.0, 1.5]: k=13 why=grad p=[ 2.007756 -0.510444]
全部关键数字复核完成


## 复核结论

**A1 实验可复现**，核心数字逐位一致（二分 39 次、Newton 比值→0.353553、多重根 0.5、固定步长表、BFGS 35 / GD 19435、LM 好/坏初值 7/13 步）。

**两处环境相关差异**（不影响结论）：
1. 5.4-B GN 坏初值 `(1,1.5)`：前 5 步轨迹逐位一致，但本环境（NumPy 2.5.1）在 k=7 因 `p[0]` 精确归零、`J^T J` 奇异而抛 `Singular matrix`；原文（NumPy 2.4.6）继续至 k=29 溢出 NaN。二者都说明 GN 无下降保证会失败，建议代码对奇异矩阵加保护。
2. 5.4-C LM 的 rho 末 1–2 个值为病态量，跨 NumPy 版本有微小差异，但最终参数/λ/标准误/scipy 对照逐位一致。